# Validation and errors

This tutorial shows the validation boundary of `gtfs2emis-py`: which inputs
are rejected, which exception type is raised and what information the message
carries. The exceptions are public API; catch the specific type rather than a
blanket `Exception` in production code.


In [ ]:
import tempfile
from pathlib import Path
from types import SimpleNamespace

import pandas as pd

from gtfs2emis_py import (
    EmissionModelError,
    EmissionWriteError,
    FleetError,
    ManifestError,
    TransportWriteError,
    emission_model,
    load_fleet,
    read_emission_manifest,
    transport_model,
    write_emission_output,
)
from gtfs2emis_py.gtfs.interpolation import parse_service_date
from gtfs2emis_py.gtfs.writer import read_transport_manifest, write_transport_output

DATA = Path.cwd() / "tests/data"
print("package imports ready")

## Service dates

Both ISO dates and the compact `YYYYMMDD` form used by GTFS `calendar.txt`
are accepted. The parser returns a `datetime.date`; it does not silently
accept arbitrary strings.


In [ ]:
for value in ("2024-01-01", "20240101"):
    print(value, "->", parse_service_date(value))

try:
    parse_service_date("01/01/2024")
except ValueError as exc:
    print(type(exc).__name__ + ":", exc)

## Fleet validation

`load_fleet` validates the columns and category vocabulary required by the
selected EF model. A missing column raises `FleetError`.


In [ ]:
try:
    load_fleet(
        pd.DataFrame({"vehicle_type": ["BUS_URBAN_D"], "fleet_composition": [1.0]}),
        ef_model="ef_brazil_cetesb",
    )
except FleetError as exc:
    print(type(exc).__name__ + ":", exc)

`ncores` must be between 1 and the machine's CPU count. The transport
model rejects an out-of-range worker count before reading shapes.


In [ ]:
try:
    transport_model(DATA / "synthetic_edge_cases.zip", parallel=True, ncores=10_000)
except ValueError as exc:
    print(type(exc).__name__ + ":", exc)

## Model validation

The EF model and process are checked against the supported sets;
`EmissionModelError` identifies the bad name and the accepted alternatives.


In [ ]:
try:
    emission_model(None, "not-a-model", pd.DataFrame(), "CO")
except EmissionModelError as exc:
    print(type(exc).__name__ + ":", exc)

try:
    emission_model(None, "ef_europe_emep", pd.DataFrame(), "CO", process="noise")
except EmissionModelError as exc:
    print(type(exc).__name__ + ":", exc)

## Manifest and write failures

A malformed or missing manifest raises `ManifestError`; an output path that
is a file rather than a directory raises `EmissionWriteError`. The transport
writer reports per-shape failures through `TransportWriteError` after it has
written the manifest.


In [ ]:
with tempfile.TemporaryDirectory() as raw:
    root = Path(raw)
    (root / "manifest.json").write_text("{not json")

    try:
        read_transport_manifest(root)
    except ManifestError as exc:
        print("transport manifest:", type(exc).__name__ + ":", exc)

    try:
        read_emission_manifest(root)
    except ManifestError as exc:
        print("emission manifest:", type(exc).__name__ + ":", exc)

    not_a_directory = root / "file.txt"
    not_a_directory.write_text("occupied")
    result = SimpleNamespace(
        emi=pd.DataFrame(
            {
                "segment_id": [0],
                "shape_id": ["S1"],
                "trip_id": ["T1"],
                "stop_sequence": [1],
                "vehicle_type": ["BUS_URBAN_D"],
                "pollutant": ["CO"],
                "process": ["hot_exhaust"],
                "value": [1.0],
                "unit": ["g"],
            }
        ),
        metadata={},
    )
    try:
        write_emission_output(result, not_a_directory)
    except EmissionWriteError as exc:
        print("write:", type(exc).__name__ + ":", exc)

    bad_table = pd.DataFrame({"shape_id": ["S1"], "payload": [object()]})
    try:
        write_transport_output(bad_table, root / "transport")
    except TransportWriteError as exc:
        print("transport write:", type(exc).__name__ + ":", exc)

## Catching the public errors

The top-level package exports the error classes, so callers can catch them
without importing implementation modules. Catch `ValueError` only for
generic argument errors that are not part of the typed error hierarchy.


In [ ]:
ERRORS = (
    FleetError,
    EmissionModelError,
    EmissionWriteError,
    ManifestError,
    TransportWriteError,
)
print([cls.__name__ for cls in ERRORS])
print("all exports have docstrings:", all(cls.__doc__ for cls in ERRORS))